In [ ]:
!pip install "huggingface-hub==2.2.0"

In [1]:
import transformers
import huggingface_hub
import tokenizers
print(transformers.__version__)
print(huggingface_hub.__version__)
print(tokenizers.__version__)

5.20.1
2.2.0
0.23.3


In [2]:
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer

W1018 23:11:14.670000 5816 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


# Создаем базу знаний

In [3]:
documents = [
    {
        "id": 1,
        "text": "Сотрудник имеет право на 28 календарных дней оплачиваемого отпуска в год.",
        "source": "hr_policy.pdf",
    },
    {
        "id": 2,
        "text": "Для оформления отпуска сотрудник должен подать заявление руководителю.",
        "source": "vacation_rules.pdf",
    },
    {
        "id": 3,
        "text": "Больничный лист оформляется при временной нетрудоспособности сотрудника.",
        "source": "sick_leave.pdf",
    },
    {
        "id": 4,
        "text": "Командировки согласуются с руководителем и оформляются до начала поездки.",
        "source": "business_trips.pdf",
    },
]

# Создаем эмбеддинги документов 

In [4]:
embedding_model = SentenceTransformer("intfloat/multilingual-e5-base")

texts = ["passage: " + doc["text"] for doc in documents]

document_vectors = embedding_model.encode(
    texts,
    normalize_embeddings=True,
    convert_to_numpy=True,
)

print("Размер матрицы:", document_vectors.shape)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

C:\Users\D\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\D\.cache\huggingface\hub\models--intfloat--multilingual-e5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Размер матрицы: (4, 768)


# Реализуем поиск

In [5]:
def retrieve(question, top_k=3):
    query = "query: " + question

    query_vector = embedding_model.encode(
        query,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )

    # Векторы нормализованы, поэтому dot product
    # эквивалентен cosine similarity.
    scores = document_vectors @ query_vector

    # Не возвращаем больше документов, чем есть в базе.
    k = min(top_k, len(documents))
    indices = np.argsort(scores)[::-1][:k]

    results = []

    for index in indices:
        results.append({
            **documents[index],
            "score": float(scores[index]),
        })

    return results

In [6]:
results = retrieve("Сколько дней отпуска положено сотруднику?", top_k=2)

for doc in results:
    print(
        f"Score: {doc['score']:.3f} | "
        f"Source: {doc['source']}"
    )
    print(doc["text"])
    print()

Score: 0.862 | Source: hr_policy.pdf
Сотрудник имеет право на 28 календарных дней оплачиваемого отпуска в год.

Score: 0.835 | Source: vacation_rules.pdf
Для оформления отпуска сотрудник должен подать заявление руководителю.



# Подключаем модель

In [7]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=(
        torch.float16
        if torch.cuda.is_available()
        else torch.float32
    ),
)

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model.to(device)
model.eval()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [16]:
def answer_question_no_rag(question):
    messages = [
        {
            "role": "system",
            "content": ("Ты специалист по трудовому законодательству."
                        "Если данных недостаточно, скажи об этом. "
                        "Не выдумывай факты. Указывай источники."
                       )
        },
        {
            "role": "user",
            "content": f"{question}"
        }
    ]
    
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096,
    ).to(device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Убираем токены входного промпта.
    new_tokens = outputs[
        :, inputs["input_ids"].shape[1]:
    ]

    answer = tokenizer.decode(
        new_tokens[0],
        skip_special_tokens=True,
    )

    return {
        "question": question,
        "answer": answer
    }

In [17]:
result = answer_question_no_rag("Сколько дней отпуска положено сотруднику?")
print("Вопрос:", result["question"])
print("Ответ:", result["answer"])

Вопрос: Сколько дней отпуска положено сотруднику?
Ответ: В соответствии с трудовым законодательством Российской Федерации, сотрудникам предоставляется оплачиваемый отпуск в размере 2 недели за каждый год работы.

Однако, если сотрудник работает более 10 лет, то ему предоставляется дополнительный отпуск в виде ежегодного оплачиваемого假期 (ЕОП) на сумму 5-10 рабочих дней в зависимости от количества проработанных лет.

Например:
- За 10 лет работы: 2 недели + 5 дней ЕОП = 7 дней
- За 15 лет работы: 2 недели + 10 дней ЕОП = 12 дней

Эти правила могут немного отличаться в разных регионах России и у некоторых работодателей, но основные нормы остаются одинаковыми.


In [18]:
def answer_question_rag(question, top_k=3):
    retrieved = retrieve(question, top_k=top_k)

    context_parts = []

    for doc in retrieved:
        context_parts.append(
            f"[Источник: {doc['source']}]\n"
            f"{doc['text']}"
        )

    context = "\n\n".join(context_parts)

    messages = [
        {
            "role": "system",
            "content": (
                "Ты помощник по корпоративным документам. "
                "Отвечай на русском языке, опираясь только на предоставленный контекст. "
                "Если данных недостаточно, скажи об этом. "
                "Не выдумывай факты. Указывай источники."
            ),
        },
        {
            "role": "user",
            "content": (
                f"Контекст:\n{context}\n\n"
                f"Вопрос: {question}"
            ),
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096,
    ).to(device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Убираем токены входного промпта.
    new_tokens = outputs[
        :, inputs["input_ids"].shape[1]:
    ]

    answer = tokenizer.decode(
        new_tokens[0],
        skip_special_tokens=True,
    )

    sources = list(dict.fromkeys(
        doc["source"] for doc in retrieved
    ))

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "retrieved_documents": retrieved,
    }

In [19]:
result = answer_question_rag("Сколько дней отпуска положено сотруднику?")
print("Вопрос:", result["question"])
print("Ответ:", result["answer"])
print("Источники:", result["sources"])

Вопрос: Сколько дней отпуска положено сотруднику?
Ответ: Согласно предоставленному контексту, сотрудник имеет право на 28 календарные дни оплачиваемого отпуска в год.
Источники: ['hr_policy.pdf', 'vacation_rules.pdf', 'sick_leave.pdf']


# Используем FAISS 

In [24]:
import faiss
import numpy as np

# Эмбеддинги уже нормализованы.
vectors = np.asarray(document_vectors, dtype="float32")

dimension = vectors.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(vectors)

In [25]:
def retrieve_faiss(question, top_k=3):
    query_vector = embedding_model.encode(
        "query: " + question,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )

    query_vector = np.asarray(
        [query_vector],
        dtype="float32",
    )

    k = min(top_k, len(documents))

    scores, indices = index.search(
        query_vector,
        k,
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue

        results.append({
            **documents[int(idx)],
            "score": float(score),
        })

    return results

In [26]:
results = retrieve_faiss("Сколько дней отпуска положено сотруднику?", top_k=2)

for doc in results:
    print(
        f"Score: {doc['score']:.3f} | "
        f"Source: {doc['source']}"
    )
    print(doc["text"])
    print()

Score: 0.862 | Source: hr_policy.pdf
Сотрудник имеет право на 28 календарных дней оплачиваемого отпуска в год.

Score: 0.835 | Source: vacation_rules.pdf
Для оформления отпуска сотрудник должен подать заявление руководителю.



In [27]:
def answer_question_rag(question, top_k=3):
    retrieved = retrieve_faiss(question, top_k=top_k)

    context_parts = []

    for doc in retrieved:
        context_parts.append(
            f"[Источник: {doc['source']}]\n"
            f"{doc['text']}"
        )

    context = "\n\n".join(context_parts)

    messages = [
        {
            "role": "system",
            "content": (
                "Ты помощник по корпоративным документам. "
                "Отвечай на русском языке, опираясь только на предоставленный контекст. "
                "Если данных недостаточно, скажи об этом. "
                "Не выдумывай факты. Указывай источники."
            ),
        },
        {
            "role": "user",
            "content": (
                f"Контекст:\n{context}\n\n"
                f"Вопрос: {question}"
            ),
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096,
    ).to(device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Убираем токены входного промпта.
    new_tokens = outputs[
        :, inputs["input_ids"].shape[1]:
    ]

    answer = tokenizer.decode(
        new_tokens[0],
        skip_special_tokens=True,
    )

    sources = list(dict.fromkeys(
        doc["source"] for doc in retrieved
    ))

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "retrieved_documents": retrieved,
    }

In [28]:
result = answer_question_rag("Сколько дней отпуска положено сотруднику?")
print("Вопрос:", result["question"])
print("Ответ:", result["answer"])
print("Источники:", result["sources"])

Вопрос: Сколько дней отпуска положено сотруднику?
Ответ: Согласно предоставленному контексту, сотрудник имеет право на 28 календарные дни оплачиваемого отпуска в год.
Источники: ['hr_policy.pdf', 'vacation_rules.pdf', 'sick_leave.pdf']
